# 📊 Visualisation et dashboard

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 2 · 9h45** · Exercice

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Construire des visualisations analytiques et communicantes
- Monter un dashboard interactif Plotly
- Y intégrer la représentativité des segments et des alertes sur les biais

In [30]:
%pip install -q fairlearn
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

Note: you may need to restart the kernel to use updated packages.
✅ Configuration prête : chaque fichier de données se télécharge au premier usage


Downloading...
From (original): https://drive.google.com/uc?id=1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r
From (redirected): https://drive.google.com/uc?id=1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r&confirm=t&uuid=f47df58a-b5ba-4f82-9663-5de68d279a44
To: c:\Users\abiol\Downloads\analyse explorative\config_projet.py

  0%|          | 0.00/6.94k [00:00<?, ?B/s]
100%|██████████| 6.94k/6.94k [00:00<00:00, 22.4MB/s]


In [32]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Crédit Azur · 20477 lignes, 20 colonnes


In [33]:
import plotly.express as px
import plotly.graph_objects as go
from config_projet import exporter_html, PALETTE
from fairlearn.metrics import MetricFrame, selection_rate, count
df = nettoyage_minimal(brut, cfg)
for col in cfg["sensibles"]:
    df[col] = df[col].fillna("inconnu")
mensuel = df.set_index(cfg["date"])[cfg["ca"]].resample("MS").sum()

## 1. Les chiffres clés
Trois indicateurs `go.Indicator` : chiffre d'affaires de 2025 avec l'évolution par rapport à 2024, part de votre segment principal, taux de décision favorable.

In [38]:
# PNB estimé des demandes de 2024 et 2025
pnb_2024 = df.loc[
    df["date_demande"].dt.year.eq(2024), "pnb_estime_eur"
].sum()

pnb_2025 = df.loc[
    df["date_demande"].dt.year.eq(2025), "pnb_estime_eur"
].sum()

# Part des crédits auto et travaux sur toute la période
part_auto_travaux = (
    df.loc[
        df["type_credit"].isin(["auto", "travaux"]),
        "pnb_estime_eur"
    ].sum()
    / df["pnb_estime_eur"].sum()
)

# Taux d'accord sur toute la période
taux_accord = df["credit_accorde"].mean()

# Trois indicateurs dans une seule figure
kpis = go.Figure()

kpis.add_trace(go.Indicator(
    mode="number+delta",
    value=pnb_2025,
    number={"valueformat": ",.0f", "suffix": " €"},
    delta={
        "reference": pnb_2024,
        "relative": True,
        "valueformat": ".1%",
        "increasing": {"color": "#0072B2"},
        "decreasing": {"color": "#D55E00"}
    },
    title={"text": "PNB estimé · demandes de 2025<br>"
                   "<sup>Évolution par rapport à 2024</sup>"},
    domain={"x": [0, 0.32], "y": [0, 1]}
))

kpis.add_trace(go.Indicator(
    mode="number",
    value=part_auto_travaux * 100,
    number={"valueformat": ".1f", "suffix": " %"},
    title={"text": "Part auto + travaux dans le PNB<br>"
                   "<sup>Ensemble de la période 2023–2025</sup>"},
    domain={"x": [0.35, 0.66], "y": [0, 1]}
))

kpis.add_trace(go.Indicator(
    mode="number",
    value=taux_accord * 100,
    number={"valueformat": ".1f", "suffix": " %"},
    title={"text": "Taux d’accord global<br>"
                   "<sup>Ensemble de la période 2023–2025</sup>"},
    domain={"x": [0.69, 1], "y": [0, 1]}
))

kpis.update_layout(
    height=260,
    template="plotly_white",
    font={"size": 14},
    margin={"t": 65, "b": 25, "l": 25, "r": 25}
)

kpis.show()

## 2. La tendance, pour les décideurs
Courbe mensuelle, tendance sur 12 mois, et l'événement du brief annoté. Le titre dit la conclusion.

In [39]:
# PNB estimé regroupé par mois de demande
mensuel = (
    df.set_index("date_demande")["pnb_estime_eur"]
    .resample("MS")
    .sum()
)

# Moyenne mobile sur les 12 derniers mois
tendance = mensuel.rolling(12).mean()

courbe = go.Figure()

courbe.add_trace(go.Scatter(
    x=mensuel.index,
    y=mensuel.values,
    mode="lines+markers",
    name="PNB estimé mensuel",
    line={"color": "#777777", "width": 2},
    hovertemplate="%{x|%m/%Y}<br>%{y:,.0f} €<extra></extra>"
))

courbe.add_trace(go.Scatter(
    x=tendance.index,
    y=tendance.values,
    mode="lines",
    name="Moyenne mobile sur 12 mois",
    line={"color": "#0072B2", "width": 4},
    hovertemplate="%{x|%m/%Y}<br>%{y:,.0f} €<extra></extra>"
))

# Date du changement de score
date_v2 = pd.Timestamp("2024-03-01")

courbe.add_shape(
    type="line",
    x0=date_v2,
    x1=date_v2,
    y0=0,
    y1=1,
    yref="paper",
    line={"color": "#D55E00", "dash": "dash", "width": 2}
)

courbe.add_annotation(
    x=date_v2,
    y=1,
    yref="paper",
    text="Déploiement du score V2 · mars 2024",
    showarrow=False,
    xanchor="left",
    yanchor="bottom"
)

courbe.update_layout(
    title="Le PNB estimé baisse de 5,0 % en 2025 par rapport à 2024",
    xaxis_title="Mois de la demande",
    yaxis_title="PNB estimé (€)",
    template="plotly_white",
    font={"size": 14},
    height=450,
    hovermode="x unified",
    legend={"orientation": "h", "y": -0.25},
    margin={"t": 95, "b": 100}
)

courbe.show()

## 3. L'analyse, pour l'équipe data
Un graphique plus riche : chiffre d'affaires mensuel par segment, avec un sélecteur interactif.

In [40]:
# PNB estimé mensuel par type de crédit
par_segment = (
    df.groupby([
        pd.Grouper(key="date_demande", freq="MS"),
        "type_credit"
    ])["pnb_estime_eur"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

couleurs = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]

detail = go.Figure()

for couleur, segment in zip(couleurs, par_segment.columns):
    detail.add_trace(go.Scatter(
        x=par_segment.index,
        y=par_segment[segment],
        mode="lines+markers",
        name=segment,
        line={"color": couleur},
        hovertemplate="%{x|%m/%Y}<br>%{y:,.0f} €<extra>%{fullData.name}</extra>"
    ))

# Menu pour afficher tous les produits ou un seul
boutons = [{
    "label": "Tous les crédits",
    "method": "update",
    "args": [{"visible": [True] * len(par_segment.columns)}]
}]

for i, segment in enumerate(par_segment.columns):
    boutons.append({
        "label": segment,
        "method": "update",
        "args": [{
            "visible": [
                j == i for j in range(len(par_segment.columns))
            ]
        }]
    })

detail.update_layout(
    title="Les crédits auto et travaux portent 83,9 % du PNB estimé",
    xaxis_title="Mois de la demande",
    yaxis_title="PNB estimé (€)",
    template="plotly_white",
    font={"size": 14},
    height=480,
    hovermode="x unified",
    updatemenus=[{
        "buttons": boutons,
        "direction": "down",
        "x": 1,
        "xanchor": "right",
        "y": 1.18
    }],
    legend={"orientation": "h", "y": -0.25},
    margin={"t": 120, "b": 100}
)

detail.show()

## 4. L'indicateur de représentativité
Pour chaque variable sensible, le ratio part observée ÷ part de référence, coloré selon le statut, avec les seuils 0,8 et 1,25.

In [ ]:
# Référence fournie avec le cours
ref = reference(SECTEUR)

# Une seule ligne par client pour mesurer la représentativité
clients = df.dropna(subset=["client_id"]).drop_duplicates("client_id")

tableaux = []

for variable in cfg["sensibles"]:
    effectifs = clients[variable].fillna("inconnu").value_counts()

    observe = (
        effectifs.rename_axis("modalite")
        .reset_index(name="effectif")
    )
    observe["part_observee"] = observe["effectif"] / len(clients)

    comparaison = observe.merge(
        ref.loc[
            ref["variable"].eq(variable),
            ["modalite", "part_reference"]
        ],
        on="modalite",
        how="outer"
    )

    comparaison["effectif"] = comparaison["effectif"].fillna(0)
    comparaison["part_observee"] = (
        comparaison["part_observee"].fillna(0)
    )

    comparaison["ratio"] = (
        comparaison["part_observee"]
        / comparaison["part_reference"].replace(0, np.nan)
    )
    comparaison["variable"] = variable

    comparaison["statut"] = comparaison["ratio"].apply(
        lambda r: "Sans référence" if pd.isna(r)
        else "Sous-représenté" if r < 0.8
        else "Surreprésenté" if r > 1.25
        else "Dans les seuils"
    )

    tableaux.append(comparaison)

representativite = pd.concat(tableaux, ignore_index=True)

rep = px.bar(
    representativite,
    x="modalite",
    y="ratio",
    color="statut",
    facet_col="variable",
    facet_col_wrap=1,
    text="statut",
    custom_data=["effectif"],
    color_discrete_map={
        "Sous-représenté": "#D55E00",
        "Surreprésenté": "#0072B2",
        "Dans les seuils": "#999999",
        "Sans référence": "#777777"
    },
    labels={
        "modalite": "Groupe",
        "ratio": "Part observée / part de référence",
        "statut": "Situation"
    },
    title="Les 65 ans et plus et l’Outre-mer sont sous-représentés"
)

rep.add_hline(y=0.8, line_dash="dash", line_color="#555555")
rep.add_hline(y=1.25, line_dash="dot", line_color="#555555")

rep.update_xaxes(matches=None, showticklabels=True)
rep.update_traces(
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "%{x}<br>Ratio : %{y:.2f}"
        "<br>Clients : %{customdata[0]:.0f}<extra></extra>"
    )
)

rep.update_layout(
    template="plotly_white",
    height=1050,
    font={"size": 14},
    margin={"t": 100, "b": 80},
    legend={"orientation": "h", "y": -0.08}
)

rep.show()

## 5. La parité de la décision
Taux de décision favorable par groupe, avec la ligne des quatre cinquièmes du meilleur taux.

In [41]:
# Taux d'accord et nombre de demandes par groupe
tableaux_parite = []

for variable in cfg["sensibles"]:
    groupes = (
        df.groupby(variable, dropna=False)["credit_accorde"]
        .agg(taux_accord="mean", effectif="size")
        .reset_index()
        .rename(columns={variable: "groupe"})
    )

    meilleur_taux = groupes["taux_accord"].max()

    groupes["variable"] = variable
    groupes["seuil"] = 0.8 * meilleur_taux
    groupes["ratio"] = groupes["taux_accord"] / meilleur_taux
    groupes["statut"] = np.where(
        groupes["ratio"] < 0.8,
        "Sous le seuil d’alerte",
        "Au-dessus ou au seuil"
    )

    tableaux_parite.append(groupes)

table_parite = pd.concat(tableaux_parite, ignore_index=True)

parite = px.bar(
    table_parite,
    x="groupe",
    y="taux_accord",
    color="statut",
    facet_row="variable",
    category_orders={"variable": list(cfg["sensibles"])},
    text="taux_accord",
    custom_data=["effectif", "ratio"],
    color_discrete_map={
        "Sous le seuil d’alerte": "#D55E00",
        "Au-dessus ou au seuil": "#0072B2"
    },
    labels={
        "groupe": "Groupe",
        "taux_accord": "Taux d’accord",
        "statut": "Situation"
    },
    title="Les taux d’accord présentent de forts écarts selon l’âge"
)

# Un seuil propre à chaque variable :
# 80 % du meilleur taux d'accord
variables = list(cfg["sensibles"])

for i, variable in enumerate(variables):
    seuil = table_parite.loc[
        table_parite["variable"].eq(variable), "seuil"
    ].iloc[0]

    parite.add_hline(
        y=seuil,
        row=len(variables) - i,
        col=1,
        line_dash="dash",
        line_color="#555555",
        annotation_text=f"Seuil d’alerte : {seuil:.1%}"
    )

parite.update_traces(
    texttemplate="%{y:.1%}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "%{x}<br>Taux d’accord : %{y:.1%}"
        "<br>Demandes : %{customdata[0]:.0f}"
        "<br>Ratio au meilleur taux : %{customdata[1]:.2f}"
        "<extra></extra>"
    )
)

parite.update_xaxes(matches=None, showticklabels=True)
parite.update_yaxes(tickformat=".0%", range=[0, 1])

parite.update_layout(
    template="plotly_white",
    height=1050,
    font={"size": 14},
    margin={"t": 100, "b": 80},
    legend={"orientation": "h", "y": -0.08}
)

parite.show()

## 6. Les alertes
Un bloc HTML qui liste en clair les groupes sous-représentés et les groupes sous la ligne des 4/5. Ne comptez pas sur la couleur seule : écrivez-le.

In [48]:
from html import escape
from IPython.display import HTML, display

# Recalcul des effectifs sur les clients uniques
clients_uniques = (
    df.dropna(subset=["client_id"])
    .drop_duplicates("client_id")
)

representativite["effectif"] = [
    int(
        clients_uniques[ligne["variable"]]
        .fillna("inconnu")
        .eq(ligne["modalite"])
        .sum()
    )
    for _, ligne in representativite.iterrows()
]

# Groupes sous-représentés
sous_rep = representativite.loc[
    representativite["ratio"] < 0.8
]

# Groupes sous le seuil de parité
ecarts = table_parite.loc[
    table_parite["ratio"] < 0.8
]

alertes = """
<section>
<h2>Les points de vigilance</h2>
<h3>Représentativité des clients</h3>
<ul>
"""

for _, ligne in sous_rep.iterrows():
    alertes += (
        f"<li>{escape(str(ligne['variable']))} — "
        f"{escape(str(ligne['modalite']))} : "
        f"ratio de {ligne['ratio']:.2f}, "
        f"sur {int(ligne['effectif'])} clients.</li>"
    )

alertes += "</ul><h3>Écarts de taux d’accord</h3><ul>"

for _, ligne in ecarts.iterrows():
    alertes += (
        f"<li>{escape(str(ligne['variable']))} — "
        f"{escape(str(ligne['groupe']))} : "
        f"{ligne['taux_accord']:.1%} d’accord, "
        f"sur {int(ligne['effectif'])} demandes. "
        f"Ratio au meilleur taux : {ligne['ratio']:.2f}.</li>"
    )

alertes += """
</ul>
<p><strong>À retenir :</strong> ces seuils sont des repères
pour identifier les écarts à approfondir.
Ils ne prouvent pas une discrimination.</p>
<p>La population nationale de référence ne correspond pas
forcément aux personnes susceptibles de demander un crédit.</p>
</section>
"""

display(HTML(alertes))


## 7. L'export HTML autonome
Assemblez le tout avec `exporter_html` (fourni) : une page pour les décideurs, une annexe pour l'équipe data. Chaque figure reçoit une description courte, utile aux lecteurs d'écran.

In [53]:
import plotly.io as pio
from pathlib import Path
import webbrowser

# Intégrer chaque graphique dans le dashboard
def graphique(figure, description, premier=False):
    return (
        "<section class='carte'>"
        + pio.to_html(
            figure,
            full_html=False,
            include_plotlyjs=True if premier else False,
            config={"responsive": True, "displaylogo": False}
        )
        + f"<p class='description'>{description}</p>"
        + "</section>"
    )


contenu = """
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Crédit Azur — De l’analyse aux décisions</title>

<style>
html { scroll-behavior: smooth; }

body {
    font-family: Arial, sans-serif;
    background: #f4f6f8;
    color: #222;
    max-width: 1200px;
    margin: auto;
    padding: 24px;
    line-height: 1.6;
    font-size: 16px;
}

h1, h2, h3 { color: #174563; }

.carte {
    background: white;
    padding: 24px;
    margin: 20px 0;
    border-radius: 10px;
}

.partie {
    background: #174563;
    color: white;
    padding: 24px;
    margin-top: 45px;
    border-radius: 10px;
    scroll-margin-top: 90px;
}

.partie h2 { color: white; margin: 0; }

.description { color: #444; }

.message {
    border-left: 5px solid #0072B2;
    background: #edf5fa;
    padding: 18px;
    margin: 20px 0;
}

nav {
    position: sticky;
    top: 0;
    background: white;
    padding: 14px;
    border-bottom: 2px solid #174563;
    z-index: 1000;
    display: flex;
    flex-wrap: wrap;
    gap: 20px;
}

nav a {
    color: #174563;
    font-weight: bold;
    text-decoration: underline;
}

li { margin-bottom: 12px; }

@media print {
    nav { position: static; }
}
</style>
</head>

<body>

<h1>Crédit Azur : comprendre la baisse et guider les décisions</h1>
<p>
Analyse de l’activité et de l’accès au crédit,
de janvier 2023 à décembre 2025.<br>
Entreprise et données fictives utilisées pour la formation.
</p>

<nav aria-label="Parties de la présentation">
</nav>


<div class="partie" id="eba">
<h2>1. Comprendre l’évolution de l’activité</h2>
<p>Contexte, chiffres clés et tendance</p>
</div>

<section class="carte">
<h3>Pourquoi cette étude ?</h3>
<p>
Le 1er mars 2024, Crédit Azur a déployé un nouveau score
d’octroi de crédit, donnant davantage de poids au contrat
et à l’ancienneté professionnelle.
</p>
<p>
Depuis, la production a baissé et des associations
questionnent l’accès au crédit des jeunes actifs.
</p>
<p><strong>Notre question :</strong>
comment évolue l’activité, quels groupes sont concernés
et quelles actions proposer ?
</p>
<p>
Une ligne représente une demande de crédit.
L’analyse porte sur 20 384 demandes après nettoyage.
</p>
</section>
"""

# PARTIE 1 — EBA
contenu += graphique(
    kpis,
    "Le PNB estimé des demandes de 2025 atteint environ 3,17 millions "
    "d’euros, soit une baisse de 5,0 % par rapport à 2024. "
    "Les deux autres indicateurs portent sur l’ensemble de 2023–2025.",
    premier=True
)

contenu += graphique(
    courbe,
    "La série mensuelle montre les variations de l’activité. "
    "La moyenne mobile sur 12 mois permet de mieux lire la tendance. "
    "La ligne verticale marque le déploiement de V2 ; "
    "elle ne démontre pas un lien de causalité."
)

contenu += """
<div class="message">
<strong>À retenir :</strong>
le PNB estimé baisse. Il faut maintenant regarder
quels produits portent l’activité et quels groupes
obtiennent moins souvent un crédit.
</div>


<div class="partie" id="yasmine">
<h2>2. Identifier les produits et les groupes concernés</h2>
<p>Produits, représentativité et taux d’accord</p>
</div>

<section class="carte">
<h3>Quels produits portent l’activité ?</h3>
<p>
Les crédits auto et travaux représentent ensemble
<strong>83,9 % du PNB estimé</strong> sur la période.
Le menu du graphique permet d’examiner chaque produit.
</p>
</section>
"""

# PARTIE 2 — YASMINE
contenu += graphique(
    detail,
    "Le PNB estimé mensuel est présenté par type de crédit. "
    "La contribution totale d’un produit dépend notamment "
    "du nombre de demandes, des accords et des montants."
)

contenu += """
<section class="carte">
<h3>Qui est représenté dans les données ?</h3>
<p>
Nous comptons ici les <strong>clients uniques</strong>,
puis nous comparons leur répartition à la référence fournie.
</p>
<p>
Un ratio inférieur à 0,8 signale une sous-représentation ;
un ratio supérieur à 1,25 signale une surreprésentation.
Ces seuils sont des repères d’analyse.
</p>
</section>
"""

contenu += graphique(
    rep,
    "La représentativité décrit la présence des groupes dans les données. "
    "Elle ne mesure pas leur taux d’accord. La population de référence "
    "doit être adaptée à la clientèle étudiée."
)

contenu += """
<section class="carte">
<h3>Qui obtient un crédit ?</h3>
<p>
Nous comptons maintenant les <strong>demandes</strong>.
Nous comparons les taux d’accord par âge, genre et région,
en gardant les effectifs pour interpréter les résultats.
</p>
</section>
"""

contenu += graphique(
    parite,
    "Pour chaque variable, la ligne correspond à 80 % du meilleur "
    "taux d’accord. Un groupe sous cette ligne déclenche une analyse "
    "plus approfondie ; cela ne prouve pas une discrimination."
)

contenu += f"<div class='carte'>{alertes}</div>"

contenu += """
<section class="carte">
<h3>Un recul particulièrement marqué chez les jeunes</h3>
<p>
Chez les 18–24 ans, le taux d’accord passe de
<strong>36,1 % avant V2 à 7,7 % après V2</strong>,
soit une baisse de <strong>28,4 points</strong>.
</p>
<p>
Sur l’ensemble de la période, le taux d’accord est de
<strong>59,6 % pour les femmes</strong> contre
<strong>67,2 % pour les hommes</strong>.
</p>
</section>

<div class="message">
<strong>À retenir :</strong>
les différences d’accès au crédit sont importantes.
Il faut comprendre leurs causes avant de décider
d’un ajustement du score.
</div>


<div class="partie" id="iswat">
<h2>3. Passer des résultats aux actions</h2>
<p>Limites, recommandations et décision proposée</p>
</div>

<section class="carte">
<h3>Ce que notre analyse ne permet pas d’affirmer</h3>
<ul>
<li>La proximité entre la baisse et V2 ne prouve pas
que le score est seul responsable.</li>
<li>Les écarts entre groupes ne prouvent pas une discrimination :
les profils financiers doivent aussi être comparés.</li>
<li>Les remboursements sont connus uniquement pour les crédits
accordés. Le taux de vrais positifs de 100 % obtenu ne permet
donc pas d’évaluer l’égalité des chances.</li>
<li>Le PNB est estimé sur la durée des crédits et regroupé
par mois de demande. Il ne correspond ni aux encaissements
mensuels ni au bénéfice net.</li>
<li>La référence de population et les petits effectifs
limitent certaines comparaisons.</li>
</ul>
</section>

<section class="carte">
<h3>Action 1 — Analyser les critères du score V2</h3>
<p><strong>Constat :</strong>
le taux d’accord des 18–24 ans recule de 28,4 points.</p>
<p><strong>Action :</strong>
étudier le rôle du contrat et de l’ancienneté professionnelle,
en comparant des profils financiers similaires.</p>
<p><strong>Responsables et délai :</strong>
équipe data et risques, sous un mois.</p>
<p><strong>Suivi :</strong>
taux d’accord par âge à profils comparables, avec les effectifs.</p>
<p><strong>Limite :</strong>
un écart brut ne suffit pas à justifier une modification du score.</p>
</section>

<section class="carte">
<h3>Action 2 — Suivre ensemble l’activité et le risque</h3>
<p><strong>Constat :</strong>
auto et travaux portent 83,9 % du PNB estimé,
mais le PNB global recule de 5,0 % en 2025.</p>
<p><strong>Action :</strong>
suivre les demandes, les accords, les montants et le PNB estimé
par produit et par canal : agence, web et courtier.</p>
<p><strong>Responsables et délai :</strong>
direction métier et risques, mise en place sous un mois,
puis suivi mensuel.</p>
<p><strong>Suivi :</strong>
ajouter les incidents de remboursement pour vérifier
si la sélection plus stricte s’accompagne d’un risque plus faible.</p>
<p><strong>Limite :</strong>
comparer des crédits ayant une durée de suivi comparable.</p>
</section>

<section class="carte">
<h3>Action 3 — Examiner régulièrement les écarts entre groupes</h3>
<p><strong>Constat :</strong>
le taux d’accord des femmes est inférieur de 7,6 points
à celui des hommes sur l’ensemble de la période.</p>
<p><strong>Action :</strong>
organiser une revue des écarts selon l’âge, le genre et la région.</p>
<p><strong>Responsables et délai :</strong>
conformité et risques, première revue sous trois mois,
puis chaque trimestre.</p>
<p><strong>Suivi :</strong>
taux d’accord, écarts, ratios, effectifs et traitement des alertes.</p>
<p><strong>Limite :</strong>
tenir compte des différences de profils et des faibles effectifs.</p>
</section>

<section class="carte">
<h3>Les indicateurs ESG retenus</h3>
<p><strong>Social :</strong>
suivre l’accès au crédit et les différences entre les groupes.</p>
<p><strong>Gouvernance :</strong>
documenter les alertes, leur analyse et les décisions prises.
Nous proposons que toutes les alertes soient examinées sous 30 jours.</p>
<p>
Les plaintes mentionnées dans le brief motivent l’étude,
mais nous ne disposons pas des réclamations détaillées.
Les données ne permettent pas de construire
un indicateur environnemental fiable.
</p>
</section>

<div class="message">
<h3>La décision proposée à Crédit Azur</h3>
<p>
<strong>Valider une analyse des critères de V2 sous un mois,
un suivi mensuel de l’activité et du risque,
et une revue trimestrielle des écarts entre groupes.</strong>
</p>
<p>
Ces vérifications permettront de décider
si le score doit être ajusté.
</p>
</div>

</body>
</html>
"""

# Nouveau nom pour conserver la version précédente
fichier = Path("dashboard_banque_presentation.html")
fichier.write_text(contenu, encoding="utf-8")

print("Dashboard enregistré :", fichier.resolve())
webbrowser.open(fichier.resolve().as_uri())

Dashboard enregistré : C:\Users\abiol\Downloads\analyse explorative\dashboard_banque_presentation.html


True

In [50]:
import webbrowser

print("Fichier enregistré ici :", fichier.resolve())
webbrowser.open(fichier.resolve().as_uri())

Fichier enregistré ici : C:\Users\abiol\Downloads\analyse explorative\dashboard_banque.html


True

## ✅ Check-list du livrable
- [ ] Le fichier s'ouvre seul dans un navigateur, sans connexion.
- [ ] Chaque graphique a un titre qui conclut et une description.
- [ ] Les alertes sont écrites, pas seulement colorées.
- [ ] Les couleurs restent lisibles pour un daltonien (palette Okabe-Ito).

💾 Enregistrez le notebook dans GitHub (« Dashboard Plotly exporté ») et envoyez le fichier HTML téléchargé au formateur.